In [1]:
%pip install pandas openpyxl pyarrow matplotlib seaborn

  Using cached pandas-3.0.6-cp313-cp313-macosx_10_13_x86_64.whl.metadata (79 kB)
  Using cached matplotlib-3.11.2-cp313-cp313-macosx_10_13_x86_64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached numpy-2.5.4-cp313-cp313-macosx_14_0_x86_64.whl.metadata (6.6 kB)
  Using cached contourpy-1.4.0-cp313-cp313-macosx_10_13_x86_64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp313-cp313-macosx_10_13_x86_64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
Using cached pandas-3.0.6-cp313-cp313-macosx_10_13_x86_64.whl (10.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.7/38.7 MB 36.7 MB/s  0:00:01m0:00:0100:01
Using cached matplotlib-3.11.2-cp313-cp313-macosx_10_13_x86_64.whl (9.5 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.4.0-cp313-cp313-macosx_10_13_x86_64.whl (297 kB)
Using cached cycler-0.

In [ ]:
import pandas as pd

sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)
df["Description"] = df["Description"].astype("string")

df.to_parquet("../data/raw/online_retail_II.parquet", index=False)

ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column Description with type object')

In [3]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

print("Cancellations:", df["Invoice"].str.startswith("C").sum())
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Price <= 0:", (df["Price"] <= 0).sum())
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Unique customers:", df["Customer ID"].nunique())
print("Exact duplicates:", df.duplicated().sum())

(1067371, 8)
Invoice                   str
StockCode                 str
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
Cancellations: 19494
Quantity <= 0: 22950
Price <= 0: 6207
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5942
Exact duplicates: 34335


In [4]:
# Stock codes that aren't the normal 5-digit product format (optionally with a letter suffix)
odd_codes = df[~df["StockCode"].str.match(r"^\d{5}[A-Za-z]*$")]
print(odd_codes["StockCode"].value_counts().head(30))

StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
CRUK              16
DCGS0076          15
TEST001           15
DCGS0003          14
gift_0001_50       8
gift_0001_40       7
DCGS0069           6
B                  6
DCGS0004           5
m                  5
gift_0001_80       4
DCGS0072           4
DCGS0066N          4
DCGS0068           3
Name: count, dtype: int64


In [5]:
check = df[df["StockCode"].str.startswith(("DCGS", "M", "PADS"))]
print(check.groupby("StockCode")["Description"].first().head(20))

StockCode
DCGS0003                  BOXED GLASS ASHTRAY
DCGS0004           HAYNES CAMPER SHOULDER BAG
DCGS0006                                 <NA>
DCGS0016                                 <NA>
DCGS0027                                 <NA>
DCGS0036                                 <NA>
DCGS0037                   KEY-RING CORKSCREW
DCGS0039                                 <NA>
DCGS0041     HAYNES MINI-COOPER PLAYING CARDS
DCGS0044              HANDZ-OFF CAR FRESHENER
DCGS0053                                 <NA>
DCGS0055                                 <NA>
DCGS0056                                 <NA>
DCGS0057                                 <NA>
DCGS0058                     MISO PRETTY  GUM
DCGS0059                                 <NA>
DCGS0060                                 <NA>
DCGS0062              ROAD-RAGE CAR FRESHENER
DCGS0066N             NAVY CUDDLES DOG HOODIE
DCGS0066P                                <NA>
Name: Description, dtype: string


In [6]:
clean = df.drop_duplicates().copy()
log = [("Raw rows", len(df)), ("After dropping exact duplicates", len(clean))]

# Keep real products: 5-digit codes (optional letter suffix) plus DCGS codes
is_product = (
    clean["StockCode"].str.match(r"^\d{5}[A-Za-z]*$")
    | clean["StockCode"].str.startswith("DCGS")
)
clean = clean[is_product].copy()
log.append(("After dropping non-product codes", len(clean)))

clean["Revenue"] = clean["Quantity"] * clean["Price"]

# How much do we lose by dropping rows without a customer?
missing_id = clean["Customer ID"].isna()
print(f"Rows missing Customer ID: {missing_id.mean():.1%}")
print(f"Revenue on rows missing Customer ID: "
      f"{clean.loc[missing_id, 'Revenue'].sum() / clean['Revenue'].sum():.1%}")

clean = clean[~missing_id].copy()
clean["Customer ID"] = clean["Customer ID"].astype("int64")
log.append(("After dropping missing Customer ID", len(clean)))

# Split cancellations from purchases
is_cancel = clean["Invoice"].str.startswith("C")
cancellations = clean[is_cancel].copy()
purchases = clean[~is_cancel & (clean["Quantity"] > 0) & (clean["Price"] > 0)].copy()
log.append(("Purchase rows", len(purchases)))
log.append(("Cancellation rows", len(cancellations)))

for step, rows in log:
    print(f"{step:<40}{rows:>12,}")

Rows missing Customer ID: 22.7%
Revenue on rows missing Customer ID: 13.6%
Raw rows                                   1,067,371
After dropping exact duplicates            1,033,036
After dropping non-product codes           1,027,212
After dropping missing Customer ID           794,223
Purchase rows                                776,577
Cancellation rows                             17,586
